# CRMLS Sold Pipeline — Numeric Distribution Review

Review the distribution of `ClosePrice`, `LivingArea`, and `DaysOnMarket` in the curated sold ledger. The cells below are path-adaptive and can run from either the project root or the `py/` directory.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

DATA_CANDIDATES = (
    Path("csv/sold_curated.csv"),
    Path("../csv/sold_curated.csv"),
)
sold_curated_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if sold_curated_path is None:
    raise FileNotFoundError(
        "Could not find csv/sold_curated.csv from the project root or py/ directory."
    )

sold_curated = pd.read_csv(sold_curated_path, low_memory=False)
REVIEW_COLUMNS = ["ClosePrice", "LivingArea", "DaysOnMarket"]
missing_columns = [column for column in REVIEW_COLUMNS if column not in sold_curated.columns]
if missing_columns:
    raise ValueError(f"sold_curated.csv is missing required EDA columns: {missing_columns}")

numeric_review = sold_curated[REVIEW_COLUMNS].apply(pd.to_numeric, errors="coerce")
print(f"Loaded {len(sold_curated):,} sold records from {sold_curated_path.resolve()}")

## Descriptive statistics

The table reports Min, Max, Mean, Median, and the requested percentiles after coercing invalid values to null.

In [ ]:
percentiles = {"P10": 0.10, "P25": 0.25, "P50": 0.50, "P75": 0.75, "P90": 0.90}
records = []
for column in REVIEW_COLUMNS:
    values = numeric_review[column].dropna()
    row = {
        "Field": column,
        "Min": values.min(),
        "Max": values.max(),
        "Mean": values.mean(),
        "Median": values.median(),
    }
    row.update({label: values.quantile(level) for label, level in percentiles.items()})
    records.append(row)

distribution_stats = pd.DataFrame(records)
formatted_stats = distribution_stats.copy()
for column in formatted_stats.columns[1:]:
    formatted_stats[column] = formatted_stats[column].map(
        lambda value: "N/A" if pd.isna(value) else f"{value:,.2f}"
    )

header = "| " + " | ".join(formatted_stats.columns) + " |"
separator = "| " + " | ".join(["---"] * len(formatted_stats.columns)) + " |"
rows = ["| " + " | ".join(map(str, row)) + " |" for row in formatted_stats.to_numpy()]
display(Markdown("\n".join([header, separator, *rows])))

## Distribution visualizations

All plots use the full non-null numeric series so outliers remain visible during data-quality review.

In [ ]:
sns.set_theme(style="whitegrid")
close_price = numeric_review["ClosePrice"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(x=close_price, ax=axes[0], color="#4C78A8")
axes[0].set_title("ClosePrice Boxplot")
axes[0].set_xlabel("ClosePrice")
sns.histplot(close_price, bins=50, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("ClosePrice Histogram")
axes[1].set_xlabel("ClosePrice")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, column, color in zip(
    axes,
    ["LivingArea", "DaysOnMarket"],
    ["#54A24B", "#E45756"],
):
    values = numeric_review[column].dropna()
    sns.histplot(values, bins=50, kde=True, ax=axis, color=color)
    axis.set_title(f"{column} Distribution")
    axis.set_xlabel(column)
plt.tight_layout()
plt.show()

## Advanced multivariate exploration

The following analysis uses a reproducible 10,000-row complete-case sample. It examines multivariate association, multicollinearity, interaction effects, and unsupervised market segmentation. Results are exploratory associations—not causal estimates.

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

BATHROOM_SOURCE = (
    "BathroomsTotal"
    if "BathroomsTotal" in sold_curated.columns
    else "BathroomsTotalInteger"
)
source_columns = {
    "ClosePrice": "ClosePrice",
    "LivingArea": "LivingArea",
    "DaysOnMarket": "DaysOnMarket",
    "rate_30yr_fixed": "rate_30yr_fixed",
    "BedroomsTotal": "BedroomsTotal",
    "BathroomsTotal": BATHROOM_SOURCE,
    "OriginalListPrice": "OriginalListPrice",
}
missing_advanced_columns = [
    source for source in source_columns.values() if source not in sold_curated.columns
]
if missing_advanced_columns:
    raise ValueError(
        "Advanced EDA columns are missing: "
        f"{sorted(set(missing_advanced_columns))}. Run py/data_curation.py first."
    )

advanced_source = pd.DataFrame(
    {name: pd.to_numeric(sold_curated[source], errors="coerce")
     for name, source in source_columns.items()}
).replace([np.inf, -np.inf], np.nan).dropna()
if len(advanced_source) < 10000:
    raise ValueError(
        f"Advanced EDA requires 10,000 complete rows; found {len(advanced_source):,}."
    )
advanced_sample = advanced_source.sample(n=10000, random_state=42).copy()
rate_cutoff = advanced_sample["rate_30yr_fixed"].median()
advanced_sample["MortgageRateBand"] = np.where(
    advanced_sample["rate_30yr_fixed"] <= rate_cutoff, "Low rate", "High rate"
)
print(f"Advanced EDA sample: {len(advanced_sample):,} rows; rate cutoff={rate_cutoff:.3f}%")

### Pairplot: multivariate association by mortgage-rate regime

The median monthly mortgage rate splits observations into low- and high-rate regimes. A corner plot and histogram diagonals keep the 10,000-row visualization tractable.

In [ ]:
pairplot_columns = ["ClosePrice", "LivingArea", "DaysOnMarket", "rate_30yr_fixed"]
pair_grid = sns.pairplot(
    advanced_sample,
    vars=pairplot_columns,
    hue="MortgageRateBand",
    hue_order=["Low rate", "High rate"],
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.35, "s": 12},
)
pair_grid.fig.suptitle("Sold-property relationships by mortgage-rate regime", y=1.02)
plt.show()

### Multicollinearity via Variance Inflation Factor (VIF)

VIF above 5 is flagged as potentially severe multicollinearity. High VIF does not invalidate prediction, but it makes individual coefficient interpretation unstable.

In [ ]:
vif_features = ["LivingArea", "BedroomsTotal", "BathroomsTotal", "OriginalListPrice"]
vif_design = sm.add_constant(advanced_sample[vif_features], has_constant="add")
vif_table = pd.DataFrame(
    {
        "Feature": vif_features,
        "VIF": [
            variance_inflation_factor(vif_design.to_numpy(), vif_design.columns.get_loc(feature))
            for feature in vif_features
        ],
    }
)
vif_table["Assessment"] = np.where(
    vif_table["VIF"] > 5, "Severe (VIF > 5)", "Acceptable"
)
vif_display = vif_table.copy()
vif_display["VIF"] = vif_display["VIF"].map(lambda value: f"{value:,.3f}")
vif_header = "| " + " | ".join(vif_display.columns) + " |"
vif_separator = "| " + " | ".join(["---"] * len(vif_display.columns)) + " |"
vif_rows = ["| " + " | ".join(map(str, row)) + " |" for row in vif_display.to_numpy()]
display(Markdown("\n".join([vif_header, vif_separator, *vif_rows])))
severe_vif = vif_table.loc[vif_table["VIF"] > 5, "Feature"].tolist()
vif_message = (
    "Severe multicollinearity detected for: **" + ", ".join(severe_vif) + "**."
    if severe_vif
    else "No feature exceeds the VIF > 5 severe-collinearity threshold."
)
display(Markdown(vif_message))

### Interaction effects with OLS

Two raw-scale interaction terms test whether the association between price and one predictor changes with another predictor. Statistical significance is evaluated using each interaction term's p-value.

In [ ]:
model_data = advanced_sample.copy()
model_data["LivingArea_x_BedroomsTotal"] = (
    model_data["LivingArea"] * model_data["BedroomsTotal"]
)
model_data["OriginalListPrice_x_rate_30yr_fixed"] = (
    model_data["OriginalListPrice"] * model_data["rate_30yr_fixed"]
)
ols_features = [
    "LivingArea",
    "BedroomsTotal",
    "OriginalListPrice",
    "rate_30yr_fixed",
    "LivingArea_x_BedroomsTotal",
    "OriginalListPrice_x_rate_30yr_fixed",
]
ols_design = sm.add_constant(model_data[ols_features], has_constant="add")
ols_result = sm.OLS(model_data["ClosePrice"], ols_design).fit()
display(ols_result.summary())

interaction_terms = ["LivingArea_x_BedroomsTotal", "OriginalListPrice_x_rate_30yr_fixed"]
interaction_tests = pd.DataFrame(
    {
        "Interaction": interaction_terms,
        "Coefficient": ols_result.params[interaction_terms].to_numpy(),
        "PValue": ols_result.pvalues[interaction_terms].to_numpy(),
    }
)
interaction_tests["SignificantAt5Pct"] = interaction_tests["PValue"] < 0.05
interaction_display = interaction_tests.copy()
interaction_display["Coefficient"] = interaction_display["Coefficient"].map(lambda x: f"{x:,.6g}")
interaction_display["PValue"] = interaction_display["PValue"].map(lambda x: f"{x:.4g}")
interaction_header = "| " + " | ".join(interaction_display.columns) + " |"
interaction_separator = "| " + " | ".join(["---"] * len(interaction_display.columns)) + " |"
interaction_rows = ["| " + " | ".join(map(str, row)) + " |" for row in interaction_display.to_numpy()]
display(Markdown("\n".join([interaction_header, interaction_separator, *interaction_rows])))

### Unsupervised clustering: four market segments

KMeans uses standardized price, living area, and market-time features. Cluster IDs are arbitrary, so business labels are assigned after inspecting each cluster profile rather than hard-coded to a numeric ID.

In [ ]:
cluster_features = ["ClosePrice", "LivingArea", "DaysOnMarket"]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(advanced_sample[cluster_features])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
advanced_sample["Cluster"] = kmeans.fit_predict(scaled_features)
advanced_sample["ClusterLabel"] = advanced_sample["Cluster"].map(lambda value: f"Cluster {value}")

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=advanced_sample,
    x="LivingArea",
    y="ClosePrice",
    hue="ClusterLabel",
    palette="tab10",
    alpha=0.55,
    s=24,
)
plt.title("KMeans market segmentation: ClosePrice vs LivingArea")
plt.tight_layout()
plt.show()

cluster_profile = (
    advanced_sample.groupby("Cluster")
    .agg(
        Records=("ClosePrice", "size"),
        MeanClosePrice=("ClosePrice", "mean"),
        MeanLivingArea=("LivingArea", "mean"),
        MeanDaysOnMarket=("DaysOnMarket", "mean"),
    )
    .sort_values("MeanClosePrice")
)
price_order = cluster_profile.index.tolist()
segment_labels = {price_order[0]: "刚需型", price_order[-1]: "豪宅型"}
middle_clusters = price_order[1:3]
slow_cluster = max(middle_clusters, key=lambda cluster: cluster_profile.loc[cluster, "MeanDaysOnMarket"])
segment_labels[slow_cluster] = "冷门/滞销型"
segment_labels[next(cluster for cluster in middle_clusters if cluster != slow_cluster)] = "投资/改善型"
cluster_profile["MarketSegment"] = [segment_labels[index] for index in cluster_profile.index]
profile_display = cluster_profile.reset_index().copy()
for column in ["MeanClosePrice", "MeanLivingArea", "MeanDaysOnMarket"]:
    profile_display[column] = profile_display[column].map(lambda value: f"{value:,.2f}")
profile_header = "| " + " | ".join(profile_display.columns) + " |"
profile_separator = "| " + " | ".join(["---"] * len(profile_display.columns)) + " |"
profile_rows = ["| " + " | ".join(map(str, row)) + " |" for row in profile_display.to_numpy()]
display(Markdown("\n".join([profile_header, profile_separator, *profile_rows])))
interpretation = ["**Data-driven market-segment interpretation:**"]
for cluster, row in cluster_profile.iterrows():
    interpretation.append(
        f"- Cluster {cluster} — **{row['MarketSegment']}**: mean price ${row['MeanClosePrice']:,.0f}, "
        f"mean area {row['MeanLivingArea']:,.0f} sq ft, mean DOM {row['MeanDaysOnMarket']:,.1f} days."
    )
interpretation.append("These labels summarize relative sample profiles and should be validated against geography and property type before operational use.")
display(Markdown("\n".join(interpretation)))